In [ ]:
# -------------------------------------------
# 第1步：启动Spark（本地运行，适配8G内存）
# 运行位置：本地 VS Code + PySpark 环境
# -------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType,LongType,DateType,DoubleType,StructType,StructField
import random
import os,sys
os.environ['PYSPARK_PYTHON']='python'
spark=SparkSession.builder\
    .appName('Bank_1000W')\
    .master('local[2]')\
    .config('spark.driver.memory','6G')\
    .config('spark.sql.adaptive.enabled','True')\
    .getOrCreate()
print('Spark启动成功!!!!!!!!!!!!!!!!!!!')

Spark启动成功!!!!!!!!!!!!!!!!!!!


In [ ]:
# -------------------------------------------
# 第2步：生成 1000万 银行交易数据
# 运行位置：本地 VS Code
# -------------------------------------------
def generate_data():
    for i in range(1,10000001):  # 循环1000万次，生成交易数据
        yield(
            i, # 交易ID（自增主键）
            f'cust_{random.randint(10000,99999)}', # 客户ID（5位随机数）
            f'acc_{random.randint(100000,999999)}', # 账户ID（6位随机数）
            round(random.uniform(10.0, 100000.0),2),  #交易金额（10元~10万元，保留2位小数）
            random.choice(['消费', '转账', '存款', '取款', '代扣']), #交易类型
            '20260405' # 分区字段：日期
        )

In [3]:
# 定义dataframe的Schema
schema=StructType([
    StructField("trans_id", LongType(), True),
    StructField("cust_id", StringType(), True),
    StructField("account_id", StringType(), True),
    StructField("trans_amt", DoubleType(), True),
    StructField("trans_type", StringType(), True),
    StructField("trans_date", StringType(), True)
])
 
# 生成DataFrame
df=spark.createDataFrame(
    generate_data(),
    schema
)
print("✅ 1000万数据生成完成")
df.printSchema()
df.show(20)

✅ 1000万数据生成完成
root
 |-- trans_id: long (nullable = true)
 |-- cust_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- trans_amt: double (nullable = true)
 |-- trans_type: string (nullable = true)
 |-- trans_date: string (nullable = true)

+--------+----------+----------+---------+----------+----------+
|trans_id|   cust_id|account_id|trans_amt|trans_type|trans_date|
+--------+----------+----------+---------+----------+----------+
|       1|cust_97845|acc_338101| 77686.96|      转账|  20260405|
|       2|cust_56750|acc_973128| 38525.36|      取款|  20260405|
|       3|cust_33951|acc_903488| 33019.17|      消费|  20260405|
|       4|cust_36879|acc_119964| 38044.76|      消费|  20260405|
|       5|cust_65438|acc_231724| 72486.98|      代扣|  20260405|
|       6|cust_22492|acc_423653| 11928.48|      转账|  20260405|
|       7|cust_92299|acc_485973|  2309.71|      消费|  20260405|
|       8|cust_64253|acc_248727| 30433.57|      消费|  20260405|
|       9|cust_77256|acc_388325|  346

In [8]:
# -------------------------------------------
# 第3步：写入 HDFS（核心！包含 分区 + 压缩）
# 运行位置：本地 VS Code
# -------------------------------------------
df.repartition(10)\
    .write\
    .mode('overwrite')\
    .partitionBy('trans_date')\
    .option('compression','snappy')\
    .parquet('hdfs://192.168.10.121:9000/user/bank/transaction')
print('✅ 1000万银行数据已写入 HDFS 完成！')

✅ 1000万银行数据已写入 HDFS 完成！


In [11]:
# -------------------------------------------
# 第4步：读取HDFS数据验证
# -------------------------------------------
df_read=spark.read.parquet('hdfs://192.168.10.121:9000/user/bank/transaction')
print(f'总共读取了:{df_read.count()}条数据---------')
df_read.show(100)

总共读取了:10000000条数据---------
+--------+----------+----------+---------+----------+----------+
|trans_id|   cust_id|account_id|trans_amt|trans_type|trans_date|
+--------+----------+----------+---------+----------+----------+
| 1210979|cust_12683|acc_511970| 69070.98|      存款|  20260405|
| 2778562|cust_76230|acc_153009| 50324.68|      存款|  20260405|
| 4018128|cust_47815|acc_341887| 31217.78|      代扣|  20260405|
| 3090030|cust_87688|acc_366617| 28424.32|      代扣|  20260405|
|  522780|cust_77198|acc_139230|  7332.17|      代扣|  20260405|
| 1459668|cust_24470|acc_276243| 69116.41|      存款|  20260405|
| 2053374|cust_57547|acc_565905| 47759.07|      消费|  20260405|
| 4712949|cust_99193|acc_236482|  85090.0|      取款|  20260405|
| 2063966|cust_69050|acc_829804| 97660.06|      消费|  20260405|
| 2221270|cust_19101|acc_681760| 69126.83|      存款|  20260405|
| 1939211|cust_66175|acc_408384| 13605.88|      代扣|  20260405|
| 1700614|cust_45073|acc_256006| 20153.12|      存款|  20260405|
| 2310369|cust_35365|a

In [1]:
spark.stop()
print("✅ Spark已关闭")

NameError: name 'spark' is not defined